# 0. Rename RHS Folders by Stim Waveform

Use this first utility block when you have a parent stimulation folder containing multiple RHS session folders. Paste the parent folder path, click **Preview Rename Plan**, and review the proposed names. Click **Apply Renames** only when the preview looks correct.

The suffix is decoded from the first recorded channel in each RHS folder that contains nonzero `stim_data`, so it can handle recordings with multiple amplifier channels and stimulation on only one channel. The folder name suffix is formatted like `A-026 stim cathodic first 100us 200uA 1Hz 30 pulses 999ms RP comp`.


### Helper code used by Block 0

The notebook is the user interface. The folder scanning, waveform decoding, and actual renaming live in [`rename_rhs_folders_by_stim_waveform.py`](rename_rhs_folders_by_stim_waveform.py).

- **Preview Rename Plan** reads each immediate RHS session folder and shows the old/new folder names without changing anything.
- **Apply Renames** applies the previewed plan only for folders marked `rename`.
- If you paste a single RHS session folder instead of its parent, the utility will rename that one folder.

- The suffix includes the detected stim channel, adds `999ms RP` from the inferred refractory gap, and adds `comp` when the RHS compliance-limit bit is present.


In [ ]:
import importlib, wideband_main_ui as wideband_ui; wideband_ui = importlib.reload(wideband_ui); wideband_ui.show_function0_rename_rhs_folders(globals())


# 2. Plot Continuous Traces (Raw / Filtered)

Use this block to view the continuous recording. **Mode** chooses the display:

- **Raw wideband** (default): no filter is applied. Long traces use a min/max display envelope, which reduces plotted points without filtering or modifying the signal. The plot carries a biphasic stim-pulse caption (amplitude and first-phase duration for the selected **Pulse** number), and the status line reports timestamp gaps. Output: `raw_wideband_*.png`.
- **Filtered**: enter `all` to keep all recorded frequencies, or a bandpass range such as `200-400 Hz`. Only samples inside the signed amplitude window (e.g. `-100 - 100 uV`) and the selected time window are shown, and the y-axis is pinned to the amplitude window. Output: `filtered_wideband_*.png`.

Check **Ignore stim (response only)** in Filtered mode to skip the stim-channel search entirely -- the old Function 4 behaviour. The amplitude default widens to `-500 - 500 uV`, no channel is starred, and the output is named `recorded_response_*.png` with a time-window label.

The preview is not saved until you click **Save PNG**. (This block absorbed the old Blocks 1 and 4 on 2026-08-20.)

### Helper code used by Block 2

Raw mode uses [`plot_rhs_raw_wideband_with_stim_legend.py`](plot_rhs_raw_wideband_with_stim_legend.py) (`plot_raw_channels_with_stim_pulse`, the biphasic pulse caption, and the RHS reader); Filtered mode uses [`plot_rhs_filtered_wideband.py`](plot_rhs_filtered_wideband.py) on top of the same reader.

- `resolve_channel_selection(...)` accepts `all`, or explicit entries like `A-014`, `A-014-16`, and `A-014, A-016`.
- `parse_time_window(...)` accepts `all` or entries like `10-20 s`.
- `parse_frequency_range(...)` parses `all` or entries like `200-400 Hz`; `parse_amplitude_range(...)` parses signed windows like `-100 - 100 uV`.
- `bandpass_filter_wideband(...)` applies a zero-phase Butterworth bandpass for numeric ranges.
- `plot_raw_channels_with_stim_pulse(...)` / `plot_filtered_channels(...)` draw the stacked traces; the stim channel is starred unless **Ignore stim** is checked.
- All loading goes through `rhs_stim.read_selected_channels(...)`, the single place a future `.rhd` reader will plug in.

In [ ]:
import importlib, wideband_main_ui as wideband_ui; wideband_ui = importlib.reload(wideband_ui); wideband_ui.show_function2_continuous_traces(globals())

# 3. Plot Stim-Triggered Bandpass Events

Use this third block to split the session into stim-triggered events based on RHS `stim_data` onsets, then place all events into one combined PNG. Each event panel is aligned to the exact RHS stim trigger at `0 s`, with a selectable pre-trigger window and post-trigger window. Events are arranged 3 per row so each row stays compact.

Function 3 uses the same response filters as Function 2: channels (`all` for every recorded channel, or explicit channels/ranges), Bandpass (`all` or a numeric range), signed amplitude window, pre time, post time, and max points. Stim onsets are detected from any recorded channel in the RHS folder that contains nonzero `stim_data`, even if that channel is not selected for display. `Train gap (ms)` controls how close stim pulses can be while still being grouped as one train/event.

Enter **Post time (ms)** as a single value like `500` to show `0-500 ms` after stim, or as a range like `20-300` to leave the first `20 ms` after stim blank in the response traces. A light grey vertical line marks the stim trigger at `0 s`. Use **Show stim current** to include or hide the red `stim_data` waveform row. The preview is not saved until you click **Save PNG**.

**Filter pipeline (Spec v2):** numeric bands are applied per event in the epoch -> blank -> filter order: each event's display window is cut with 500 ms of padding, stim pulses (-1/+5 ms) and any response-blank window are linearly interpolated away, the epoch is bandpassed zero-phase, and the padding is trimmed. The continuous trace is never filtered, so saturating stim artifact cannot ring through the recording. For quantitative work use **Block 6**.

### Helper code used by Block 3

This block uses [`plot_rhs_stim_triggered_events.py`](plot_rhs_stim_triggered_events.py) for stim-triggered event detection and plotting, and [`wideband_function3_ui.py`](wideband_function3_ui.py) for the compact notebook UI.

- `build_stim_triggered_events(...)` groups nearby `stim_data` pulses into train/event onsets.
- `plot_stim_triggered_events_grid(...)` creates one combined grid plot with 3 stim events per row, aligns all panels to stim trigger time `0 s`, draws a light grey trigger line, and adds the matching red `stim_data` waveform below the selected response channels when **Show stim current** is enabled.
- `Pre time (ms)` controls how much data before the trigger is shown; `Post time (ms)` controls the displayed post-trigger response window.
- `default_stim_events_grid_output_path(...)` names the single combined event PNG.


In [ ]:
import importlib, wideband_main_ui as wideband_ui; wideband_ui = importlib.reload(wideband_ui); wideband_ui.show_function3_stim_triggered_events(globals())


Def:
train gap (ms) controls how close stim pulses can be while still being grouped as one stimulation train/event. For example, for a 100 Hz train, pulses are about 10 ms apart. With Train gap (ms) = 12, those 5 pulses are grouped into one event. In Function 3, pre/post time is relative to each stim trigger: Pre time = 100 ms and Post time = 500 ms shows -100 to +500 ms around each trigger; Post time = 20-300 ms leaves the first 20 ms after trigger blank in the response traces.


# 5. Power Analysis

Use this block to quantify LFP/field-potential power changes without relying on spike-band activity. It runs the **Pre/Post neuromodulation** comparison: clean recording before the first stimulation sample against clean recording after the last stimulation sample, per band, with bootstrap confidence intervals. Click **Save PNG + CSV** to save both outputs inside the selected data folder.

For **event-locked** power use Block 6 (session analysis; epochs, blanks and filters per spec), or `batch_run_wideband_main_ui.py --power-mode event` for the scripted version.

### Helper code used by Block 5

This block uses [`plot_rhs_power_analysis.py`](plot_rhs_power_analysis.py) for power computation and [`wideband_function5_power_ui.py`](wideband_function5_power_ui.py) for the compact notebook UI.

- `Bands` accepts lines like `theta 4-8` or `gamma 30-80`.
- `Pre/Post neuromodulation` auto-detects the first and last nonzero `stim_data` samples, excludes the stimulation block, and compares sliding-window band power before vs after stimulation.
- `Stim-triggered events` uses `Baseline (ms)`, `Post (ms)`, `Train gap (ms)`, and `Blank (ms)` to analyze paired event windows after pulse artifact blanking.
- Heatmaps use a fixed dB color scale, and the CSV includes mean power, dB change, event/window counts, and bootstrap confidence intervals.


In [ ]:
import importlib, wideband_main_ui as wideband_ui; wideband_ui = importlib.reload(wideband_ui); wideband_ui.show_function5_power_analysis(globals())


# 6. Session Stimulation Analysis (Spec v2)

Use this block on a **session parent folder** (one sub-folder per RHS run) rather than one run. It answers the gating question first -- how much of each record is usable and what analysis window survives the stimulation artifact -- and only then runs the secondary analyses on the conditions that survive.

- **Validate** lists every run before anything else runs: pulses commanded (`settings.xml`) vs detected on the stim marker, compliance flag, empirical rail level and % samples railed per channel, block assignment and exclusions. Nothing is written.
- **Generate Preview** at stage *Recovery* measures the artifact recovery time per trial on raw epochs (threshold = max(3 x baseline SD, 100 uV), quiet 20 ms), derives the earliest usable latency per channel x amplitude (P90 of recovery + margin) and a verdict (early_ok / late_only / unusable), and renders figures 1-3. Stage *Full* adds epoch -> blank -> filter band power and RMS per trial, the three labelled comparisons (within-epoch, block vs no-stim baseline, across amplitude), drift, models (linear vs sigmoid, AIC/BIC), spatial decay, compliance, the channel random-effect model, log-normal checks and the shuffle control. Preview writes nothing.
- **Save Bundle** writes every table, figure and the metadata JSON atomically into `<session folder>/stim_analysis/`, plus per-run CSVs next to each run's `.rhs` files (untick the box to keep run folders untouched).

Rules baked in: the continuous trace is never filtered; the high-pass must be >= 1 Hz; every plot has fixed axis and colour limits; every caption states n trials retained/rejected, the blanking window and the filter; baseline and post windows are paired by event id and cropped to equal length.


### Helper code used by Block 6

This block uses the [`stim_analysis/`](stim_analysis/) package (`load_rhs`, `validate`, `epoch`, `recovery`, `metrics`, `stats`, `models`, `figures`, `secondary`, `pipeline`) and [`wideband_function6_session_ui.py`](wideband_function6_session_ui.py) for the compact notebook UI. The same pipeline runs headless with `/usr/local/bin/python3 run_stim_analysis.py "<session folder>"`.

- `Epoch (ms)` / `Baseline (ms)` / `Late (ms)` are relative to each pulse onset; epochs are cut with `Pad (ms)` of extra data on each side for filtering and trimmed afterwards. Neighbouring pulses inside the pad are blanked too.
- `k x SD`, `Floor (uV)`, `Quiet (ms)` define the recovery threshold and the quiet run that ends the artifact; `Recovery quantile` (0.9 = P90; 0.5 reproduces a median rule) plus `Blank margin` set each condition's post-window start; `Post length (ms)` its length.
- `Bands` uses the Function 5 grammar (`delta 1-4`, one per line or `;`-separated). `Bootstrap` resamples (>= 1000 for final runs; lower for a quick look). `Trace uA` picks the amplitudes shown in figure 3.
- `Baseline run` and `Stim channel` default to auto (no-stim run = the earliest run without pulses; stim channel = the one carrying `stim_data`). `Impedance CSV` is an optional explicit override; the default is the impedance stored in each RHS header.
- Self-test without data: `/usr/local/bin/python3 -m stim_analysis.selftest`.


In [ ]:
import importlib, wideband_main_ui as wideband_ui; wideband_ui = importlib.reload(wideband_ui); wideband_ui.show_function6_session_analysis(globals())


# 7. Evoked Response Sweep (RHD, external stimulator)

Use this block for sessions where the **stimulus came from an external source-meter (Keithley 2400) and the Intan only recorded**, which are saved as `.rhd` rather than `.rhs`. Functions 0-6 cannot read those files at all, and there is no stim marker channel to read pulse times from.

- **Pulse times are recovered from the amplifier trace.** ANALOG-IN and DIGITAL-IN were disabled, so nothing recorded the trigger. The known train -- 50 pulses, 5 ms wide, starting about 10 s in -- is fitted to the signal as a comb, then refined by least squares. A plain amplitude threshold does not work here: it locks onto the ongoing rhythm instead of the stimulus.
- **Every run says whether its timing can be trusted.** The fit reports how far the winning alignment beats every other alignment; a run where the stimulus is too small to see is flagged rather than given plausible-looking wrong epochs. Periods established by confident runs are then used to re-time the weak ones, and those runs are flagged too.
- **Three measures per run and channel:** per-pulse evoked deflection (peak-to-peak and latency, referenced both to the pre-train period and to the quiet late gap between pulses), band power during the train vs an equal baseline, and post-train change.
- **Coupling is tested for, not assumed away.** Latency, how much energy outlasts the 5 ms pulse, linearity in current and polarity symmetry are reported per run so a large deflection can be judged rather than taken at face value.

**Mode** *Single run* checks one run folder in seconds; *Whole session* reads every run below the folder (about 1.9 GB for session 20260819). Preview writes nothing; **Save Bundle** writes `verdict.txt`, `runs.csv`, `conditions.csv` and four figures atomically into `<folder>/evoked_sweep/`.


### Helper code used by Block 7

This block uses the [`evoked_sweep/`](evoked_sweep/) package (`config`, `naming`, `load`, `pulses`, `metrics`, `artifact`, `pipeline`, `figures`, `summary`) on top of [`rhd_reader.py`](rhd_reader.py), with [`wideband_function7_evoked_ui.py`](wideband_function7_evoked_ui.py) for the notebook UI.

- `Folder` takes a session folder (whole sweep) or a single run folder (quick check). Paths with spaces can be pasted as-is.
- `Channels` defaults to `all`, which means every contact whose header impedance is under 1 Mohm. In session 20260819 that is B-017, B-018 and B-021 early on; B-020 and B-023 join partway through when their impedance drops, so channel availability legitimately varies between runs.
- `Pulses` and `Width (ms)` are the known protocol, used as priors and as correctness checks -- a run that does not return that many pulses at a consistent period is flagged.
- `Max runs` limits a whole-session pass for a quick look; 0 means all.
- Amplitude comes from the folder name: an underscore is the decimal point (`-0_02mA` = -0.02 mA) and a bare leading zero means tenths (`-02mA` = -0.2 mA). Names are parsed literally; suspected magnitude mislabels are flagged, never corrected.
- Band power is reported twice: comb-excluded (valid for every band) and gap-based (blank for delta and theta, whose periods do not fit in the ~200 ms gap between pulses).
- Self-test without data: `/usr/local/bin/python3 -m evoked_sweep.selftest`, and `/usr/local/bin/python3 rhd_reader_selftest.py` for the reader.


In [ ]:
import importlib, wideband_main_ui as wideband_ui; wideband_ui = importlib.reload(wideband_ui); wideband_ui.show_function7_evoked_response(globals())